In [22]:
#库包安装以及停用词文档下载
!pip install pkuseg tqdm pandas scikit-learn
!wget https://raw.githubusercontent.com/goto456/stopwords/master/cn_stopwords.txt -O stopwords.txt


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
--2025-11-03 07:03:48--  https://raw.githubusercontent.com/goto456/stopwords/master/cn_stopwords.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 100.65.138.175
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|100.65.138.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4717 (4.6K) [text/plain]
Saving to: ‘stopwords.txt’

stopwords.txt       100%[===================>]   4.61K  --.-KB/s    in 0s      

2025-11-03 07:03:48 (48.4 MB/s) - ‘stopwords.txt’ saved [4717/4717]



In [23]:
import os
import re
import json
import pkuseg
from tqdm import tqdm

# ===== 配置路径 =====
DATASET_DIR = "CSTS"
OUT_DIR = "CSTS_preprocessed"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== 分词器与停用词 =====
seg = pkuseg.pkuseg(model_name="default")
with open("stopwords.txt", "r", encoding="utf-8") as f:
    stopwords = set([w.strip() for w in f if w.strip()])

# ===== 文本清洗与分词函数 =====
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\u4e00-\u9fa5a-zA-Z0-9 ]", "", text)
    return text.strip()

def tokenize(text):
    words = seg.cut(clean_text(text))
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

# ===== 处理单个数据集 =====
def process_dataset(name):
    dataset_path = os.path.join(DATASET_DIR, name)
    if not os.path.isdir(dataset_path):
        return
    out_path = os.path.join(OUT_DIR, name)
    os.makedirs(out_path, exist_ok=True)

    for split in ["train.json", "dev.json", "test.json"]:
        in_file = os.path.join(dataset_path, split)
        if not os.path.exists(in_file):
            continue
        with open(in_file, "r", encoding="utf-8") as f:
            data = [json.loads(line) for line in f]

        new_data = []
        for d in tqdm(data, desc=f"Processing {name}/{split}"):
            s1 = tokenize(d.get("sentence1", ""))
            s2 = tokenize(d.get("sentence2", ""))
            label = d.get("label", None)
            new_data.append({"sentence1": s1, "sentence2": s2, "label": label})

        out_file = os.path.join(out_path, split)
        with open(out_file, "w", encoding="utf-8") as f:
            for d in new_data:
                f.write(json.dumps(d, ensure_ascii=False) + "\n")

for sub in os.listdir(DATASET_DIR):
    if os.path.isdir(os.path.join(DATASET_DIR, sub)):
        process_dataset(sub)

print("✅ 全部预处理完成，结果已保存到:", OUT_DIR)


✅ 全部预处理完成，结果已保存到: CSTS_preprocessed


In [1]:
#传统方法计算文本相似度（TF-IDF baseline）
import json
import os
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score

DATASET = "CSTS_preprocessed/LCQMC"  # 可切换为任意子集
SPLIT = "test.json"

# === 读取数据 ===
data = [json.loads(line) for line in open(os.path.join(DATASET, SPLIT), "r", encoding="utf-8")]
s1 = [d["sentence1"] for d in data]
s2 = [d["sentence2"] for d in data]
labels = [int(d["label"]) for d in data]

# === 构建TF-IDF向量 ===
corpus = s1 + s2
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(corpus)
v1, v2 = vectors[:len(s1)], vectors[len(s1):]

# === 计算余弦相似度 ===
scores = cosine_similarity(v1, v2).diagonal()  # 对应行对相似度
preds = (scores > 0.5).astype(int)  # 阈值可调

# === 输出评估指标 ===
acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds)
print(f"✅ TF-IDF Baseline on {DATASET}: ACC={acc:.4f}, F1={f1:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: 'CSTS_preprocessed/LCQMC/test.json'